<a href="https://colab.research.google.com/github/ZMvega/Estructuras_de_datos_y_laboratorio/blob/main/Laboratorio1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import numpy as np

FILAS = 100_000
COLUMNAS = 100_000
TAMANO_BLOQUE = 1_000
NOMBRE_ARCHIVO = "matriz_bits_separada.dat"
FILAS_A_MOSTRAR = 5



def escribir_matriz_bits_separada(ruta, filas, columnas, tamano_bloque=TAMANO_BLOQUE):
    columnas_bytes = (columnas // 8) + 1

    matriz_disco = np.memmap(ruta, dtype="uint8", mode="w+",
                              shape=(filas, columnas_bytes))

    for i in range(0, filas, tamano_bloque):
        filas_actuales = min(tamano_bloque, filas - i)

        #booleanos
        bloque_bits = np.random.randint(0, 2, size=(filas_actuales, columnas), dtype=bool)
        #a bytes
        bloque_comprimido = np.packbits(bloque_bits, axis=1)
        #saltos
        columna_saltos = np.full((filas_actuales, 1), 10, dtype="uint8")
        bloque_final = np.hstack((bloque_comprimido, columna_saltos))
        matriz_disco[i:i + filas_actuales, :] = bloque_final

    matriz_disco.flush()
    del matriz_disco

    print("Matriz empaquetada creada con exito.\n")
    print(f"Dimensiones reales en disco: ({filas}, {columnas_bytes})")
    print(f"Peso final del archivo: {os.path.getsize(ruta) / (1024**3):.2f} GB")


#exportar, muestra 5 filas
def mostrar_filas_bloques(ruta, columnas, filas_a_mostrar=FILAS_A_MOSTRAR):
    bytes_datos_por_fila = (columnas + 7) // 8
    bytes_por_fila_en_disco = bytes_datos_por_fila + 1

    with open(ruta, "rb") as f:
        for fila in range(filas_a_mostrar):
            f.seek(fila * bytes_por_fila_en_disco)
            datos = f.read(bytes_datos_por_fila)

            bits = []
            for byte in datos:
                for k in range(7, -1, -1):
                    if len(bits) < columnas:
                        bits.append((byte >> k) & 1)

            print(",".join(str(b) for b in bits))


if __name__ == "__main__":
    escribir_matriz_bits_separada(NOMBRE_ARCHIVO, FILAS, COLUMNAS, TAMANO_BLOQUE)
    print("\nMostrando las primeras filas como bloques:\n")
    mostrar_filas_bloques(NOMBRE_ARCHIVO, COLUMNAS, FILAS_A_MOSTRAR)

Matriz empaquetada creada con exito.

Dimensiones reales en disco: (100000, 12501)
Peso final del archivo: 1.16 GB

Mostrando las primeras filas como bloques:

0,1,1,0,0,1,1,1,0,1,1,1,1,0,0,1,1,1,1,1,0,0,0,0,0,1,1,1,1,1,0,1,1,0,1,0,1,1,1,1,1,1,1,1,0,1,0,1,0,1,1,1,1,0,0,1,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,1,1,1,1,0,1,0,0,1,1,0,0,0,1,0,1,1,1,1,0,1,0,1,0,0,0,1,0,1,0,1,0,0,1,1,0,1,0,0,0,1,0,1,0,1,0,1,1,1,1,0,1,1,1,0,0,0,0,1,0,0,1,0,1,0,0,0,1,1,1,0,0,1,1,0,0,0,0,1,1,0,1,1,0,0,1,0,0,0,1,0,1,1,0,1,1,0,0,1,0,0,0,1,1,1,1,0,1,0,0,0,0,1,1,0,1,0,0,0,0,0,1,1,1,0,1,1,0,0,0,0,0,1,0,1,0,0,1,1,1,1,1,1,0,1,1,0,0,1,1,1,1,1,0,0,1,1,1,1,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,1,0,1,1,0,1,0,0,1,0,0,0,1,0,1,0,0,1,1,0,0,1,1,1,1,1,1,0,0,1,0,0,0,0,1,0,1,1,0,1,0,1,1,0,0,0,0,1,1,1,1,1,1,1,0,0,0,1,1,0,1,0,1,1,0,1,1,1,0,0,0,0,1,0,1,0,0,0,1,1,1,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,1,1,0,1,1,0,1,0,1,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,1,0,1,0,1,0,0,1,1,0,0,0,1,0,1,1,1,1,0,0,1,1,1,0,0,1,0,1,1,0,1,1,1,1,0,0,1,1,1,1,

Es importante aclarar que el peso de las columnas se encuentra en bits, 12.501 = 100000 columnas.

Se muestran las primeras 5 filas en consola.